# Generate a dataset for fine-tuning SBERT 

### Step 1: Generate weak labels 
Label subset of the data using rule-based regexing, if keywords are present, then assign the label. 

### Step 2: Prepare the data as matching pairs
Build positive pairs from the weak labels, by pairing comments according to the similarity of the labels assigned. Use soft-confidence scores to account for the fact that the labelling in step 1 is weak. 

In [ ]:
import pandas as pd
import numpy as np
import re
import random

import pickle

from collections import Counter
from itertools import combinations 

from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset
from sentence_transformers import InputExample

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks
from comments_saver import CommentsSaver

/opt/miniconda3/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read the remote database of comments

In [2]:
cs = CommentsSaver(env='dev')
df = cs.read_all()

Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.


In [3]:
print('df shape:', df.shape)

df shape: (30994, 11)


### Split a small fraction of the dataset to label 

In [4]:
# Assume df is your DataFrame and 'application_id' is the column to group by
# Just splittling a tiny fraction (1%) for the sake of the example
gss = GroupShuffleSplit(n_splits=1, train_size=0.01, random_state=42)

# Get the application_id groups
groups = df['application_id']

# Split the indices
train_inds, test_inds = next(gss.split(df, groups=groups))

# Create train and test DataFrames
df_train = df.iloc[train_inds].reset_index(drop=True)

In [5]:
print('df training set shape:', df_train.shape)

df training set shape: (384, 11)


### Pre-process the data 

In [6]:
model_checkpoint = "../outputs/nlp_fine_tuning/distilbert-base-uncased"
nlp = NLP_Tasks(model_checkpoint)

# remove place names, numbers and non-ASCII characters
# df = nlp.remove_place_names(df=df, column='comment_text')
# df = nlp.remove_numbers(df=df, column='cleaned_comment_text')

# remove non-ASCII characters
df_train = nlp.remove_non_ascii(df=df_train, column='comment_text')

# split text on newlines
df_train = nlp.split_text_on_newline(df=df_train, column='comment_text', filter_empty=True, filter_short=True, min_length=5)

Device set to use mps:0


In [7]:
print('df training set shape:', df_train.shape)

df training set shape: (942, 11)


In [8]:
# issue_topic_map = {
#     r"\b(character|appearance|visual|aesthetic|look|design)\b": "Out of character with area",
#     r"\b(scale|dominance|size|bulk|mass)\b": "Scale of development",
#     r"\b(density|overcrowd|crammed|packed)\b": "Density of development",
#     r"\b(heritage|historic|listed building|conservation)\b": "Impact on heritage",
#     r"\b(housing mix|unit mix|family housing)\b": "Housing mix wrong",
#     r"\b(affordable|unaffordable|costly|low income)\b": "Not affordable",
#     r"\b(amenities|local services|convenience)\b": "Impact on local amenities",
#     r"\b(schools|clinics|utilities|infrastructure|parks)\b": "Impact on social infrastructure",
#     r"\b(highway|road network|transport network|infrastructure)\b": "Impact on transport infrastructure",
#     r"\b(safety|accident risk|danger|hazard)\b": "Impact on transport safety",
#     r"\b(air quality|pollution|emissions|smoke|dust|fumes)\b": "Increased air pollution",
#     r"\b(trees|green space|wildlife|nature|open space)\b": "Loss of green space",
#     r"\b(parking|car space|vehicle access|loading)\b": "Loss of parking",
#     r"\b(consultation|feedback|engagement|public opinion)\b": "Inadequate public consultation",
#     r"\b(privacy|overlook|view into|windows facing)\b": "Loss of privacy",
#     r"\b(daylight|sunlight|shadow|overshadow)\b": "Loss of light",
#     r"\b(disabled|accessibility|wheelchair)\b": "Disabled access",
#     r"\b(traffic|congestion|vehicle volume)\b": "Impact on transport infrastructure",
#     r"\b(drainage|flood|runoff)\b": "Drainage and flooding",
#     r"\b(noise|dust|fumes|construction impact)\b": "Increased air pollution",
#     r"\b(community impact|local effect)\b": "Impact on community",
#     r"\b(economy|jobs|sustainability|affordability)\b": "Impact on economy",
#     r"\b(government policy|statutory|regulations|planning rules)\b": "Against government policy",
#     r"\b(local plan|neighbourhood plan)\b": "Against local plan",
#     r"\b(national policy|regional plan|framework)\b": "Against planning policies",
#     r"\b(previous planning|past decisions|appeals)\b": "Prior planning decisions",
#     r"\b(noise disturbance|ongoing noise)\b": "Increased noise pollution",
#     r"\b(hazardous|toxic|chemical)\b": "Hazardous materials",
#     r"\b(smell|odour|odors)\b": "Increased smells",
#     r"\b(ugly|unsightly|unattractive|unappealing)\b": "Unaesthetic",
#     r"\b(landscaping|greenscaping)\b": "Landscaping issues",
#     r"\b(access road|vehicle access)\b": "Impact on road access",
#     r"\b(archaeology|historical site)\b": "Archaeology",
#     r"\b(solar panels|solar power)\b": "Solar panels",
#     r"\b(applicant background|morals|personal views)\b": "Applicants character",
#     r"\b(view loss|blocked view)\b": "Loss of view",
#     r"\b(property value|house price drop)\b": "Loss of property value",
#     r"\b(trade|competition|business impact)\b": "Impact on trade",
#     r"\b(object|petition|resistance)\b": "Volume of objections",
#     r"\b(construction work|development noise|building works)\b": "Construction disturbance",
#     r"\b(damage|property damage|cracks|subsidence)\b": "Fear of property damage",
#     r"\b(maintenance|upkeep|repair neglect)\b": "Lack of maintenance",
#     r"\b(boundary|property line|easement|rights)\b": "Boundary disputes",
#     r"\b(neighbour dispute|neighbor conflict)\b": "Neighbourhood disputes",
#     r"\b(contractual|covenant|legal obligations)\b": "Contractual obligations",
#     r"\b(right of way|access rights|footpath dispute)\b": "Right of way"
# }

### Define the terms for the regex matching - heuristic 

In [ ]:
issue_topic_map = {
    "Out of character with area": r"\b(character|incongruous|appearance|architectural|visual|aesthetic|look|design|style|context|blend|beauty|charm|out of keeping)\b",
    "Scale of development": r"\b(scale|bulk|height|dominance|overbearing|oversized|mass|tall|overdevelop|over develop)\b",
    "Density of development": r"\b(density|dense|overdevelopment|crammed|crowded|too many units|congested|overcrowding)\b",
    "Impact on heritage": r"\b(heritage|listed|conservation|historic|preserv)\b",
    "Housing mix wrong": r"\b(mix|family|bedroom|flats only|unit type|tenure|local housing needs|social housing|council housing)\b",
    "Not affordable": r"\b(affordable|affordability|social housing|council housing|luxury|market rate)\b",
    "Impact on local amenities": r"\b(amenity|impact on|local high street|commercial units)\b",
    "Impact on social infrastructure": r"\b(school|no public benefits|gp|clinic|healthcare|park|playground|community service|childcare|local resources)\b",
    "Impact on transport infrastructure": r"\b(traffic|highway|transport|parking|car|bus|congestion|accessibility)\b",
    "Impact on garbage infrastructure": r"\b(waste|garbage|bin|rubbish|trash|collection|disposal|sanitation|litter|dump|overflow|vermin|rodent)\b",
    "Impact on transport safety": r"\b(safety|dangerous|accident|pedestrian|footpath|crossing|collision|vehicle hazard)\b",
    "Increased air pollution": r"\b(air quality|pollution|dust|emission|fumes|vehicle exhaust)\b",
    "Increased noise pollution": r"\b(noise|loud|disturbance|sound|construction noise|disruption)\b",
    "Loss of green space": r"\b(green space|trees|garden|open space|vegetation|greenery|nature|green spaces|outdoor space)\b",
    "Damage to natural environment": r"\b(environment|wildlife|biodiversity|habitat|nature|ecology)\b",
    "Loss of parking": r"\b(parking|car space|vehicle storage|no place to park|permit|number of vehicles)\b",
    "Impact on water infrastructure": r"\b(flood|drainage|surface water|sewer|runoff|waterlog|waterpipe|water pipe|water flow|water board)\b",
    "Inadequate public consultation": r"\b(consultation|not informed|residents ignored|no say|ballot|engagement|misled)\b",
    "Loss of privacy": r"\b(privacy|overlook|see into|direct view|window intrusion|overlooking)\b",
    "Loss of light": r"\b(light|shadow|sunlight|daylight|block sun|loss of light|overshadow)\b",
    "Impact on visual amenity": r"\b(visual|eyesore|ugly|disturbance|view ruined|inappropriate look)\b",
    "Disabled access": r"\b(disabled|wheelchair|mobility|accessible|step-free|inclusive design)\b",
    "Archaeology": r"\b(archaeolog|historic site|ancient|dig|relic)\b",
    "Right of way": r"\b(right of way|footpath|access route|public access|pathway blocked)\b",
    "Contractual obligations": r"\b(covenant|agreement|legal obligation|contract|binding terms)\b",
    "Applicants character": r"\b(developer trust|past record|reputation|history|untrustworthy|previous projects)\b",
    "Loss of view": r"\b(view|scenic|vista|lookout|blocked view)\b",
    "Loss of property value": r"\b(value|property price|devaluation|resale)\b",
    "Impact on trade": r"\b(trade|business|shop|retail|commercial impact|livelihood)\b",
    "Volume of objections": r"\b(objections|petition|local opposition|many comments|campaign)\b",
    "Construction disturbance": r"\b(building work|dust|disruption|noise from works|timeline)\b",
    "Lack of maintenance": r"\b(maintenance|neglect|disrepair|upkeep|deterioration)\b",
    "Neighbourhood disputes": r"\b(neighbour|dispute|argument|row|resident disagreement)\b",
    "Not meeting environmental standards": r"\b(environmental standard|carbon|eco|sustainable|green standard|BREEAM|passivhaus|climate)\b",
    "Concerns about precarity": r"\b(precarity|uncertain future|unstable|temporary housing|no guarantee)\b",
    "Misleading application": r"\b(misleading|inaccurate|dishonest|bait and switch|changed plans|false information|lack measurements)\b",
    "Replacement accommodation not suitable": r"\b(replacement housing|unsuitable|not like for like|no garden|flat instead of house)\b",
    "Residents forced to leave area": r"\b(evict|move away|displaced|no return|gentrification|moved out)\b",
    "Developer previously built below standard dwellings": r"\b(poor quality|shoddy work|track record|bad build|low standard)\b",
    "Increase in antisocial behaviour": r"\b(antisocial|crime|drugs|violence|unruly|unsafe)\b",
    "Will improve safety": r"\b(safety improvement|better lighting|secure|surveillance|safe area)\b",
    "Area needs regeneration": r"\b(regeneration|revitalise|modernise|redevelop|run-down)\b",
    "Better replacement accomodation": r"\b(better housing|modern homes|improved living|higher quality|new build benefit)\b",
    "Area needs more housing": r"\b(housing need|shortage|waiting list|supply|more homes)\b",
    "Meets standards": r"\b(standard met|compliant|satisfies code|approved|safe design|meets regulation)\b",
    "In character with local area": r"\b(is in keeping|matches area|context appropriate|fits in|neighbourhood style)\b",
    "Private only spaces": r"\b(private|exclusive|residents only|no public access|gated|segregated)\b",
    "Lack of space for social distancing": r"\b(crowded|too close|no distancing|compact|post-COVID)\b",
    "Accomodation unsuitable for vulnerable people": r"\b(vulnerable|elderly|disabled|high rise harm|inaccessible)\b",
    "Boundary disputes": r"\b(boundary wall|dividing wall|property line|encroachment|land dispute)\b",
}


### Apply the regex matching algorithm 

In [10]:
def assign_topics(comment):
    matched_topics = []
    for topic, pattern in issue_topic_map.items():
        if re.search(pattern, comment, flags=re.IGNORECASE):
            matched_topics.append(topic)
    return matched_topics if matched_topics else ["No match"]

# Apply to DataFrame
df_train['assigned_topics'] = df_train['comment_text'].apply(assign_topics)

In [ ]:
# Save the DataFrame with assigned topics to a CSV file
df_train[['stance', 'comment_text', 'assigned_topics']].to_csv('../outputs/SBERT_pairs/df_train.csv', index=False)

### Can also use stemming and fuzzy matching 
Stemming is when you match the stem word, and don't about the case. Fuzzy matching is when you allow for variation, including stemming and similar words/terms. 
I've found that this over macthes the terms, and everyhting gets assigned every topic label. 

In [21]:
# import re
# from nltk.stem import PorterStemmer
# from fuzzysearch import find_near_matches

# ps = PorterStemmer()

# # Preprocess keywords into stemmed versions
# stemmed_issue_topic_map = {
#     topic: [ps.stem(word) for word in re.findall(r'\w+', pattern)]
#     for topic, pattern in issue_topic_map.items()
# }

# def assign_topics(comment):
#     matched_topics = []
#     comment_lower = comment.lower()
#     comment_words = re.findall(r'\w+', comment_lower)
#     stemmed_comment_words = [ps.stem(word) for word in comment_words]

#     for topic, keywords in stemmed_issue_topic_map.items():
#         for stemmed_kw in keywords:
#             for word in comment_words:
#                 if ps.stem(word) == stemmed_kw:
#                     matched_topics.append(topic)
#                     break
#                 # Fuzzy match with max 1 edit distance
#                 if find_near_matches(stemmed_kw, word, max_l_dist=1):
#                     matched_topics.append(topic)
#                     break
#             else:
#                 continue
#             break

#     return list(set(matched_topics)) if matched_topics else ["No match"]

# # Apply to DataFrame
# df_train['assigned_topics'] = df_train['comment_text'].apply(assign_topics)


In [22]:
df_train[['assigned_topics', 'comment_text']]

,assigned_topics,comment_text
0,[No match],I have some issues I would like to raise
1,[No match],Re : the proposed side extension.
2,[Loss of green space],I am not sure how this extension will allow ac...
3,[Increased noise pollution],I would like to ensure that this extension mai...
4,"[Out of character with area, Impact on heritag...","The proposed additions to the top, back and si..."
...,...,...
937,[No match],More errors and grief from planning.
938,[No match],All trouble and errors from the department. So...
939,[No match],Incompetence errors in the description about a...
940,[No match],So object to all


In [23]:
# Count rows where 'assigned_topics' is exactly ['No match']
no_match_count = df_train['assigned_topics'].apply(lambda x: x == ['No match']).sum()
total_count = df_train.shape[0]
no_match_percent = (no_match_count / total_count) * 100

print(f"Percentage of 'No match' comments: {no_match_percent:.2f}%")

Percentage of 'No match' comments: 25.69%


In [24]:
# Flatten all topic lists into a single list
all_topics = [topic for sublist in df_train['assigned_topics'] for topic in sublist if topic != "No match"]

# Count occurrences of each topic
topic_counter = Counter(all_topics)

# Total number of rows (not total topic mentions)
total_rows = df_train.shape[0]

# Calculate percentage of rows associated with each topic
topic_percentages = {topic: (count / total_rows) * 100 for topic, count in topic_counter.items()}

# Display results
print("Percentage of rows associated with each topic:")
for topic, percent in sorted(topic_percentages.items(), key=lambda x: -x[1]):
    print(f"{topic}: {percent:.2f}%")


Percentage of rows associated with each topic:
Out of character with area: 23.25%
Impact on transport infrastructure: 22.19%
Loss of parking: 11.68%
Loss of light: 10.62%
Loss of privacy: 9.87%
Increased noise pollution: 8.70%
Loss of green space: 8.39%
Scale of development: 7.86%
Impact on local amenities: 7.22%
Impact on social infrastructure: 7.22%
Density of development: 7.22%
Construction disturbance: 4.88%
Impact on visual amenity: 4.46%
Housing mix wrong: 4.25%
Impact on heritage: 4.03%
In character with local area: 4.03%
Impact on garbage infrastructure: 4.03%
Increased air pollution: 3.50%
Impact on transport safety: 3.18%
Impact on trade: 2.97%
Damage to natural environment: 2.97%
Meets standards: 1.91%
Inadequate public consultation: 1.70%
Not affordable: 1.70%
Accomodation unsuitable for vulnerable people: 1.70%
Loss of property value: 1.59%
Loss of view: 1.38%
Private only spaces: 1.38%
Volume of objections: 1.06%
Lack of space for social distancing: 1.06%
Disabled access:

In [25]:
# return count of items with mutliple matched topics
multiple_matches = df_train[df_train['assigned_topics'].apply(lambda x: len(x) > 1)]
print(f"Number of comments with multiple matched topics: {len(multiple_matches)}")

Number of comments with multiple matched topics: 442


In [ ]:
df_train_no_match = df_train[df_train['assigned_topics'].apply(lambda x: x == ['No match'])].reset_index(drop=True)
df_train_no_match[['stance', 'comment_text', 'assigned_topics']].to_csv('../outputs/SBERT_pairs/df_train_no_match.csv', index=False)

### Create positive and negative pairs for contrastive loss 

In [18]:
# # filter out 'no match' or empty topic lists
# df_train = df_train[df_train['assigned_topics'].apply(lambda x: x != ['No match'] and len(x) > 0)].reset_index(drop=True)

# # build a mapping from topic -> list of comment_texts
# topic_to_comments = defaultdict(list)

# for _, row in df_train.iterrows():
#     for topic in row['assigned_topics']:
#         topic_to_comments[topic].append(row['comment_text'])

# # Generate InputExampe pairs for contrastive loss 
# positive_pairs = []
# for topic, comment in topic_to_comments.items():
#     # create all unique pairs within the same topic
#     for combo in combinations(set(comment), 2):
#         positive_pairs.append(InputExample(texts=[combo[0], combo[1]], label=1.0))

# print(f'Generated {len(positive_pairs)} positive pairs for contrastive loss.')

In [19]:
# import random

# # Sample some negative pairs (different topics)
# all_comments = df_train['comment_text'].tolist()
# negative_pairs = []

# for _ in range(len(positive_pairs)):
#     c1, c2 = random.sample(all_comments, 2)
#     # Ensure they don't share a topic
#     topics1 = set(df_train[df_train['comment_text'] == c1]['assigned_topics'].values[0])
#     topics2 = set(df_train[df_train['comment_text'] == c2]['assigned_topics'].values[0])
#     if topics1.isdisjoint(topics2):
#         negative_pairs.append(InputExample(texts=[c1, c2], label=0.0))

# print(f"Generated {len(negative_pairs)} negative pairs")

In [20]:
# train_examples = positive_pairs + negative_pairs

In [21]:
# Filter out rows with no valid topics
# I don't want my results to be too noisy - so filter out those comments that have no topics assigned or only 'No match'
df_train = df_train[df_train['assigned_topics'].apply(lambda x: 'No match' not in x and len(x) > 0)]

Use soft labels for the pairs - this accounts for the fact that the data was quite weakly labelled using the heuristic approach of regex matching. 

### Soft-labelled positive pairs 
* all topics shared -> label=1
* all but one topic shared -> label=0.75
* all but two topics shared -> label=0.5

Then generate the same number of negative pairs - which is useful for contrastive loss training. I just sample a random set of these - as could potentially generate an enormous number of these, since n choose 2 etc. 
### negative pairs 
* 0 shared topics -> label=0

In [ ]:
# # Create positive pairs with soft labels
# positive_pairs = []
# comments = df_train[['comment_text', 'assigned_topics']].values.tolist()

# for (text1, topics1), (text2, topics2) in combinations(comments, 2):
#     shared_topics = set(topics1).intersection(set(topics2))
#     if shared_topics:
#         score = min(1.0, 0.25 * len(shared_topics))  # Cap at 1.0
#         positive_pairs.append(InputExample(texts=[text1, text2], label=score))

# print(f"Generated {len(positive_pairs)} soft-labeled positive pairs.")

# # Optional: Generate balanced negative pairs (label = 0.0)
# negative_pairs = []
# texts_and_topics = df_train[['comment_text', 'assigned_topics']].values.tolist()

# while len(negative_pairs) < len(positive_pairs):
#     text1, topics1 = random.choice(texts_and_topics)
#     text2, topics2 = random.choice(texts_and_topics)
#     if text1 != text2 and set(topics1).isdisjoint(set(topics2)):
#         negative_pairs.append(InputExample(texts=[text1, text2], label=0.0))

# print(f"Generated {len(negative_pairs)} hard negative pairs.")

# # Combine for training
# train_examples = positive_pairs + negative_pairs


Generated 68786 soft-labeled positive pairs.
Generated 68786 hard negative pairs.


In [12]:
# Build all valid comment-topic pairs
comments = df_train[['comment_text', 'assigned_topics']].values.tolist()
positive_pairs = []

def compute_soft_label(topics1, topics2):
    set1, set2 = set(topics1), set(topics2)
    if set1 == set2:
        return 1.0
    shared = set1.intersection(set2)
    total = max(len(set1.union(set2)), 1)
    diff_count = total - len(shared)
    
    if len(shared) == 0:
        return None  # will be handled as negative
    elif diff_count == 1:
        return 0.75
    elif diff_count == 2:
        return 0.5
    else:
        return None  # skip low-similarity cases

# Generate positive pairs with refined labels
for (text1, topics1), (text2, topics2) in combinations(comments, 2):
    score = compute_soft_label(topics1, topics2)
    if score is not None:
        positive_pairs.append(InputExample(texts=[text1, text2], label=score))

print(f"Generated {len(positive_pairs)} refined positive pairs.")

# Generate balanced hard negatives
negative_pairs = []
while len(negative_pairs) < len(positive_pairs):
    text1, topics1 = random.choice(comments)
    text2, topics2 = random.choice(comments)
    if text1 != text2 and set(topics1).isdisjoint(set(topics2)):
        negative_pairs.append(InputExample(texts=[text1, text2], label=0.0))

print(f"Generated {len(negative_pairs)} negative pairs.")

# Combine for training
train_examples = positive_pairs + negative_pairs

Generated 50738 refined positive pairs.
Generated 50738 negative pairs.


In [13]:
for i, ex in enumerate(train_examples[:5]):
    print(f"Pair {i+1} - Label: {ex.label}")
    print("Text 1:", ex.texts[0][:150], "...")
    print("Text 2:", ex.texts[1][:150], "...")
    print("-" * 80)


Pair 1 - Label: 1.0
Text 1: I have some issues I would like to raise ...
Text 2: Re : the proposed side extension. ...
--------------------------------------------------------------------------------
Pair 2 - Label: 1.0
Text 1: I have some issues I would like to raise ...
Text 2: Council should refuse for the following reasons: ...
--------------------------------------------------------------------------------
Pair 3 - Label: 1.0
Text 1: I have some issues I would like to raise ...
Text 2: 2. This is connected to previous planning applications. Clearly the plan all along was to carve this up into subsized units. ...
--------------------------------------------------------------------------------
Pair 4 - Label: 1.0
Text 1: I have some issues I would like to raise ...
Text 2: The Barnet Society objects to this proposal. ...
--------------------------------------------------------------------------------
Pair 5 - Label: 1.0
Text 1: I have some issues I would like to raise ...
Text 2: De

In [ ]:
# Save train_examples to a file
with open('../outputs/SBERT_pairs/train_examples.pkl', 'wb') as f:
    pickle.dump(train_examples, f)